### Mapping flooding risk

In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import rasterio
from rasterio.transform import from_origin
from rasterio.io import MemoryFile

# ==========================================
# STEP 1: Create Dummy Flood Map (Raster)
# ==========================================
# We create a virtual 5x5 grid (like a GeoTIFF) representing water depth in meters.
# We use the Dutch coordinate system (EPSG:28992).
raster_data = np.array([
    [0.0, 0.0, 0.1, 0.5, 1.2],
    [0.0, 0.0, 0.2, 0.8, 1.5],
    [0.0, 0.1, 0.4, 1.0, 1.8],
    [0.2, 0.3, 0.6, 1.2, 2.0],
    [0.5, 0.8, 1.0, 1.5, 2.5]
], dtype=np.float32)

# Define the geographic location of the top-left corner of this grid 
# (Typical Dutch RD coordinates: X=100000, Y=400000) and pixel size (5 meters)
transform = from_origin(100000, 400000, 5, 5)

# ==========================================
# STEP 2: Create Dummy Portfolio (BAG Points)
# ==========================================
# We create three dummy properties with exact X/Y coordinates inside our grid area.
portfolio_data = {
    'Address_ID': ['P-001', 'P-002', 'P-003'],
    'Asset_Value_EUR': [350000, 420000, 280000],
    'geometry': [
        Point(100002, 399998), # High up, should be 0.0m flood depth
        Point(100012, 399988), # In the middle, moderate flood depth
        Point(100022, 399982)  # Bottom right, severe flood depth
    ]
}

# Convert to a GeoDataFrame and assign the Dutch CRS (EPSG:28992)
gdf_portfolio = gpd.GeoDataFrame(portfolio_data, crs="EPSG:28992")

# ==========================================
# STEP 3: The Mapping (Point Sampling)
# ==========================================
# We extract the coordinates from our portfolio to "pierce" the raster map.
coords = [(x, y) for x, y in zip(gdf_portfolio.geometry.x, gdf_portfolio.geometry.y)]

# We use a MemoryFile here just so you don't have to save a real file to your hard drive.
# In reality, you would just use: with rasterio.open("path_to_your_floodmap.tif") as src:
with MemoryFile() as memfile:
    with memfile.open(
        driver='GTiff', height=5, width=5, count=1, dtype=str(raster_data.dtype),
        crs='EPSG:28992', transform=transform
    ) as dataset:
        dataset.write(raster_data, 1) # Write our dummy grid into the virtual file
        
        # This is the core mapping function: sampling the raster at our specific points
        sampled_values = list(dataset.sample(coords))

# Extract the values from the sampling result and add them as a new column
gdf_portfolio['Flood_Depth_m'] = [val[0] for val in sampled_values]

# ==========================================
# STEP 4: Calculate Financial Impact (Dummy Rule)
# ==========================================
# Let's apply a very simple dummy vulnerability curve:
# For every meter of water, you lose 20% of the asset's value.
def calculate_damage(row):
    damage_percentage = min(row['Flood_Depth_m'] * 0.20, 1.0) # Max 100% loss
    return row['Asset_Value_EUR'] * damage_percentage

gdf_portfolio['Expected_Loss_EUR'] = gdf_portfolio.apply(calculate_damage, axis=1)

# Print the final ESG Stress Test tabular output
print(gdf_portfolio.drop(columns='geometry'))

  Address_ID  Asset_Value_EUR  Flood_Depth_m  Expected_Loss_EUR
0      P-001           350000            0.0           0.000000
1      P-002           420000            0.4       33600.000501
2      P-003           280000            2.0      112000.000000


In [2]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Polygon
import rasterio
from rasterio.transform import from_origin
from rasterstats import zonal_stats

# ==========================================
# STEP 1: Create Dummy Flood Map (Raster)
# ==========================================
# A 5x5 grid representing water depth in meters.
raster_data = np.array([
    [0.0, 0.0, 0.1, 0.5, 1.2],
    [0.0, 0.0, 0.2, 0.8, 1.5],
    [0.0, 0.1, 0.4, 1.0, 1.8],
    [0.2, 0.3, 0.6, 1.2, 2.0],
    [0.5, 0.8, 1.0, 1.5, 2.5]
], dtype=np.float32)

# Set the top-left coordinate (X=100000, Y=400000) and pixel size (5x5 meters)
transform = from_origin(100000, 400000, 5, 5)

# ==========================================
# STEP 2: Create Dummy Postcodes (Polygons)
# ==========================================
# Instead of points, we draw boxes (polygons) representing two PC6 postcodes.
# Postcode A covers the top-left (mostly dry area).
# Postcode B covers the bottom-right (heavy flood area).
portfolio_data = {
    'Postcode': ['1011AA', '1011AB'],
    'Total_Asset_Value_EUR': [1500000, 2000000], # Sum of all houses in the postcode
    'geometry': [
        Polygon([(100000, 400000), (100010, 400000), (100010, 399990), (100000, 399990)]), 
        Polygon([(100015, 399985), (100025, 399985), (100025, 399975), (100015, 399975)])
    ]
}

# Convert to a GeoDataFrame and set the Dutch CRS (EPSG:28992)
gdf_postcodes = gpd.GeoDataFrame(portfolio_data, crs="EPSG:28992")

# ==========================================
# STEP 3: The Mapping (Zonal Statistics)
# ==========================================
# We use rasterstats to overlay the polygons onto the array and extract the 'max' value.
# (We pass the raw numpy array and the affine transform so it knows how to align them)
stats = zonal_stats(
    gdf_postcodes.geometry, 
    raster_data, 
    affine=transform, 
    stats="max",       # We ask for the maximum flood depth in that postcode
    nodata=-9999       # Standard nodata value just in case
)

# Extract the 'max' values from the dictionary list returned by zonal_stats
gdf_postcodes['Max_Flood_Depth_m'] = [stat['max'] for stat in stats]

# ==========================================
# STEP 4: Calculate Financial Impact
# ==========================================
# Using the same dummy rule: 20% loss per meter of water, capped at 100%
def calculate_zonal_damage(row):
    # If there is no flood data (e.g., polygon is outside the map), default to 0
    if pd.isna(row['Max_Flood_Depth_m']):
        return 0
    
    damage_percentage = min(row['Max_Flood_Depth_m'] * 0.20, 1.0)
    return row['Total_Asset_Value_EUR'] * damage_percentage

gdf_postcodes['Expected_Loss_EUR'] = gdf_postcodes.apply(calculate_zonal_damage, axis=1)

# Print the final output (dropping the geometry column for a cleaner table)
print(gdf_postcodes.drop(columns='geometry'))

  Postcode  Total_Asset_Value_EUR  Max_Flood_Depth_m  Expected_Loss_EUR
0   1011AA                1500000                0.0                0.0
1   1011AB                2000000                2.5          1000000.0
